In [ ]:
import os
import sys
import torch
import torchvision

from torch import optim
# from torchvision.transforms import GaussianBlur, Resize

from optical_volume import optics, scene, utils
# from py_wave_propagator import torch_volume_prop as propagator

from matplotlib import pyplot as plt

In [ ]:
# example: functioning
WL = 500e-9
OBJ_SPACING = [100e-9, 100e-9, 100e-9]
SHAPE = [600, 600, 100]
NA = 0.7
DIST_PM_IM = 1.43e-3

TILE_LEN = 12e-6
PM_HEIGHT = 100e-9
PM_RI = 1.46
PM_SIG = 0.6

N_BG = 1.51
PAD = 512
FOC_PLANE_VAR = 0.1e-8

MAG = 40
OBJ_IMG_FACTOR = 20
EFF_MAG = int(40/20)

IMG_SPACING = [OBJ_SPACING[i]*OBJ_IMG_FACTOR for i in range(2)]
DIGITAL_PX = 6e-6
SUM_PATCH = int(DIGITAL_PX/IMG_SPACING[0])

DEVICE = 'cuda:1'


sphere1 = scene.shapes.Sphere(torch.tensor((20e-6, 45e-6, 3e-6)).float(), torch.tensor((5e-6)), torch.tensor(1.46))
sphere2 = scene.shapes.Sphere(torch.tensor((30e-6, 20e-6, 5e-6)).float(), torch.tensor((3e-6)), torch.tensor(1.46))
    
vol = scene.CollectionVolume(SHAPE, OBJ_SPACING, N_BG)
vol.add([sphere1, sphere2])

vol_prop = optics.VolumePropagator(WL, OBJ_SPACING, N_BG)

# plt.imshow(vol.forward()[:, :, 50].detach())
# plt.colorbar()
# plt.show()

lens = optics.F4ImagingSystem(WL, OBJ_SPACING, SHAPE, NA)
mag_obj = optics.Magnification(MAG)  # physical: spacing x MAG, amplitude / MAG
resample = optics.Resample(EFF_MAG)  # numerical: EFF_MAG x more samples -> IMG_SPACING
pm_obj = optics.BinaryPhaseMask(WL, TILE_LEN, IMG_SPACING, [x*EFF_MAG for x in SHAPE], PM_HEIGHT, PM_SIG, PM_RI)
px_int_obj = optics.PixelIntegrator(SUM_PATCH)

# tile_size_px = torch.ceil(torch.tensor(TILE_LEN/SPACING[0])).int() # 25, sqaure cells in voxels
# num_tiles_x, num_tiles_y = torch.ceil(SHAPE[0]/tile_size_px).int(), torch.ceil(SHAPE[1]/tile_size_px).int() # 16x16, number of tiles
# pm_obj = optics.CustomPhaseMask(WL, TILE_LEN, SPACING, SHAPE, torch.rand(num_tiles_x, num_tiles_y).float()*1e-6, PM_SIG, PM_RI)

prop_obj_space = optics.FreeSpacePropagator(WL/N_BG, OBJ_SPACING[:2], SHAPE[2]*OBJ_SPACING[2]/2 - FOC_PLANE_VAR, direction='backward', padding=PAD)
prop_to_mask = optics.FreeSpacePropagator(WL, IMG_SPACING, DIST_PM_IM, direction='backward', padding=PAD)
prop_to_sensor = optics.FreeSpacePropagator(WL, IMG_SPACING, DIST_PM_IM, direction='forward', padding=PAD)
prop_to_sensor.distance = prop_to_mask.distance  # one physical distance, trained together

cwfs_basic = optics.system.OpticalSystem((
        prop_obj_space, # centring + displacing focus plalne
        lens, # imaging the wavefield
        mag_obj, # magnification MAG
        resample, # sampling at IMG_SPACING
        prop_to_mask, # image plane -> mask plane
        pm_obj, # applying the mask
        prop_to_sensor, # mask plane -> sensor (image plane)
        optics.Intensity(), # |E|^2 at the sensor
        px_int_obj # integrate intensity over each camera pixel
    ))

vol.to(DEVICE)
cwfs_basic.to(DEVICE)

exit_field = vol_prop.forward(vol.forward())

ref_field = cwfs_basic(torch.ones(SHAPE[:2]).type(torch.complex64).to(DEVICE))
cwfs_basic.fields.clear()
spn_field = cwfs_basic(exit_field.type(torch.complex64))

img_ref = ref_field.detach().to('cpu')
img_spn = spn_field.detach().to('cpu')

fig, axs = plt.subplots(1, 2, figsize=(15, 5))

cm0 = axs[0].imshow(img_ref/img_ref.max(), cmap='gray', vmin=0, vmax=1.2)
plt.colorbar(cm0, ax=axs[0])

cm1 = axs[1].imshow(img_spn/img_ref.max(), cmap='gray', vmin=0, vmax=1.2)
plt.colorbar(cm1, ax=axs[1])

plt.show()

In [ ]:
display(cwfs_basic.state_dict())
print('------------------')
for params in cwfs_basic.parameters():
    display(params)

In [ ]:
display(vol.state_dict())
print('------------------')
for params in vol.parameters():
    display(params)

In [ ]:
# optimizer = optim.Adam(list(vol.parameters()) + list(cwfs_basic.parameters()), lr=1e-3)

optimizer = optim.Adam([
    {"params": vol.parameters(), "lr": 1e-2},
    {"params": [prop_to_mask.distance, pm_obj.height], "lr": 1e-3}
])

In [ ]:
vol.to('cpu'), cwfs_basic.to('cpu')